# Module 6: Policy Gradient — REINFORCE from Scratch

## Learning Objectives
By completing this notebook, you will:
1. Understand why policy gradient methods optimize the policy directly rather than a value function
2. Implement the REINFORCE algorithm with a softmax policy network in PyTorch
3. Measure the variance-reduction effect of a learned baseline
4. Interpret policy entropy as a signal of exploration during training

## Prerequisites
- Module 5: Deep Q-Networks (experience replay, target networks)
- Basic PyTorch: `nn.Module`, `optim`, `backward()`
- Familiarity with CartPole-v1 observation and action spaces

## Estimated Time: 15 minutes

---

## Why Policy Gradients?

DQN learns a **value function** and extracts a policy implicitly (`argmax Q`). This works well for discrete actions but breaks for continuous or high-dimensional action spaces. Policy gradient methods learn the **policy directly** — a mapping from states to action probabilities — making them naturally suited for both discrete and continuous control.

The core idea: if an action led to high reward, increase its probability. If it led to low reward, decrease it. The **policy gradient theorem** gives us the gradient of expected return with respect to policy parameters:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

where $G_t = \sum_{k=0}^{T-t} \gamma^k r_{t+k}$ is the **discounted return from step $t$**.

REINFORCE is the Monte Carlo estimate of this gradient: collect a full episode, compute returns, and update.

In [ ]:
# Purpose: Import all dependencies and configure reproducibility
# Key Concept: Seeding numpy, torch, and gymnasium ensures identical runs

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cpu')  # CartPole trains fastest on CPU
print(f"PyTorch {torch.__version__} | Gymnasium {gym.__version__}")

## 1. The Policy Network

The policy network maps observations to **action logits**. A softmax converts logits to probabilities. We sample from this distribution during training — this is what makes the policy **stochastic**, enabling exploration without epsilon-greedy.

For CartPole-v1: 4 inputs (cart position, cart velocity, pole angle, pole angular velocity), 2 outputs (push left / push right).

In [ ]:
# Purpose: Define the policy network architecture
# Key Concept: The network outputs logits; Categorical distribution handles softmax + sampling

class PolicyNetwork(nn.Module):
    """
    Softmax policy network for discrete action spaces.

    Parameters
    ----------
    obs_dim : int
        Dimension of the observation space.
    act_dim : int
        Number of discrete actions.
    hidden_dim : int
        Width of the single hidden layer.
    """

    def __init__(self, obs_dim: int, act_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim)
        )

    def forward(self, obs: torch.Tensor) -> Categorical:
        """Return a Categorical distribution over actions."""
        logits = self.net(obs)
        return Categorical(logits=logits)


# Quick sanity check
env_test = gym.make('CartPole-v1')
obs_dim = env_test.observation_space.shape[0]   # 4
act_dim = env_test.action_space.n               # 2
env_test.close()

policy_test = PolicyNetwork(obs_dim, act_dim)
dummy_obs = torch.zeros(1, obs_dim)
dist = policy_test(dummy_obs)
print(f"Action probabilities for zero observation: {dist.probs.detach().numpy().round(3)}")
print(f"Sampled action: {dist.sample().item()}")

## 2. Computing Discounted Returns

After collecting an episode, we compute $G_t$ for every step. The key is computing this **backwards** — $G_{T-1} = r_{T-1}$, $G_{T-2} = r_{T-2} + \gamma G_{T-1}$, and so on. This avoids an $O(T^2)$ nested loop.

**Normalizing returns** (subtract mean, divide by std) is a practical trick that stabilizes gradients without changing the optimal policy.

In [ ]:
# Purpose: Compute discounted returns for a full episode
# Key Concept: Backward scan is O(T); normalizing reduces gradient variance

def compute_returns(rewards: list, gamma: float = 0.99) -> torch.Tensor:
    """
    Compute normalized discounted returns for a single episode.

    Parameters
    ----------
    rewards : list of float
        Reward at each timestep.
    gamma : float
        Discount factor.

    Returns
    -------
    torch.Tensor
        Normalized returns, shape (T,).
    """
    # Step 1: Backward scan to accumulate discounted returns
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)

    returns = torch.tensor(returns, dtype=torch.float32)

    # Step 2: Normalize to zero mean and unit variance
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    return returns


# Verify with a toy episode: reward 1 every step, 5 steps, gamma=0.9
toy_rewards = [1.0, 1.0, 1.0, 1.0, 1.0]
toy_returns = compute_returns(toy_rewards, gamma=0.9)
print("Toy returns (normalized):", toy_returns.numpy().round(3))
print("Note: earlier steps should have higher raw returns (more future reward)")

## 3. REINFORCE without Baseline

The REINFORCE loss is:

$$\mathcal{L}(\theta) = -\sum_t \log \pi_\theta(a_t | s_t) \cdot G_t$$

We negate because PyTorch minimizes, but the gradient theorem calls for maximization.

**Policy entropy** measures how uncertain the policy is about which action to take:

$$H(\pi) = -\sum_a \pi(a|s) \log \pi(a|s)$$

High entropy early in training (exploratory), decreasing entropy as the policy becomes confident.

In [ ]:
# Purpose: Collect one episode and compute the REINFORCE policy gradient
# Key Concept: log_prob * return is the per-step policy gradient contribution

def collect_episode(env: gym.Env, policy: PolicyNetwork) -> dict:
    """
    Run one episode and record log-probabilities, rewards, and entropy.

    Returns
    -------
    dict with keys: 'log_probs', 'rewards', 'entropy', 'total_reward'
    """
    obs, _ = env.reset()
    log_probs, rewards, entropies = [], [], []

    done = False
    while not done:
        obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        dist = policy(obs_tensor)

        action = dist.sample()
        log_probs.append(dist.log_prob(action))
        entropies.append(dist.entropy())

        obs, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(reward)
        done = terminated or truncated

    return {
        'log_probs': log_probs,
        'rewards': rewards,
        'entropy': entropies,
        'total_reward': sum(rewards)
    }


def reinforce_update(episode: dict, optimizer: optim.Optimizer, gamma: float = 0.99) -> float:
    """
    Perform one REINFORCE gradient update (no baseline).

    Returns
    -------
    float
        Scalar policy loss (for logging).
    """
    returns = compute_returns(episode['rewards'], gamma)
    log_probs = torch.stack(episode['log_probs'])

    # Policy gradient loss: negate because we maximize expected return
    loss = -(log_probs * returns).sum()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

## 4. REINFORCE with Baseline

The high variance of REINFORCE comes from using raw returns $G_t$ as the credit signal. A **baseline** $b(s_t)$ — typically the state value $V(s_t)$ — is subtracted to center the signal without introducing bias:

$$\mathcal{L}(\theta) = -\sum_t \log \pi_\theta(a_t | s_t) \cdot (G_t - b(s_t))$$

The baseline is a separate network trained to predict $G_t$ with MSE loss. This is the first step toward Actor-Critic.

In [ ]:
# Purpose: Define baseline (value) network and combined update
# Key Concept: The baseline reduces variance of the policy gradient without adding bias

class BaselineNetwork(nn.Module):
    """
    Value function baseline V(s) that predicts discounted return.

    Parameters
    ----------
    obs_dim : int
        Dimension of the observation space.
    hidden_dim : int
        Width of the single hidden layer.
    """

    def __init__(self, obs_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.net(obs).squeeze(-1)


def reinforce_baseline_update(
    episode: dict,
    policy: PolicyNetwork,
    baseline: BaselineNetwork,
    policy_opt: optim.Optimizer,
    baseline_opt: optim.Optimizer,
    gamma: float = 0.99
) -> tuple:
    """
    Perform REINFORCE update with a learned value baseline.

    Returns
    -------
    tuple of (policy_loss, baseline_loss)
    """
    returns = compute_returns(episode['rewards'], gamma)
    log_probs = torch.stack(episode['log_probs'])

    # Collect observations from the episode to pass through baseline
    # (stored during collect_episode_with_obs)
    obs_tensor = episode['obs_tensor']  # shape (T, obs_dim)
    values = baseline(obs_tensor)       # shape (T,)

    # Advantage = return - baseline prediction
    advantages = returns - values.detach()  # detach so baseline grad doesn't affect policy

    # Policy loss uses advantage instead of raw return
    policy_loss = -(log_probs * advantages).sum()

    # Baseline loss: MSE between predicted value and actual return
    baseline_loss = nn.functional.mse_loss(values, returns)

    # Update policy
    policy_opt.zero_grad()
    policy_loss.backward()
    policy_opt.step()

    # Update baseline
    baseline_opt.zero_grad()
    baseline_loss.backward()
    baseline_opt.step()

    return policy_loss.item(), baseline_loss.item()


def collect_episode_with_obs(env: gym.Env, policy: PolicyNetwork) -> dict:
    """
    Like collect_episode but also stores observations for baseline training.
    """
    obs, _ = env.reset()
    log_probs, rewards, entropies, obs_list = [], [], [], []

    done = False
    while not done:
        obs_tensor = torch.tensor(obs, dtype=torch.float32)
        obs_list.append(obs_tensor)

        dist = policy(obs_tensor.unsqueeze(0))
        action = dist.sample()
        log_probs.append(dist.log_prob(action))
        entropies.append(dist.entropy())

        obs, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(reward)
        done = terminated or truncated

    return {
        'log_probs': log_probs,
        'rewards': rewards,
        'entropy': entropies,
        'obs_tensor': torch.stack(obs_list),
        'total_reward': sum(rewards)
    }


print("Policy and baseline network classes defined.")

## 5. Training Both Variants

We train REINFORCE (no baseline) and REINFORCE+Baseline side-by-side, using identical hyperparameters and seeds, then compare reward curves and policy entropy.

In [ ]:
# Purpose: Training loop for both REINFORCE variants
# Key Concept: Same seeds ensure fair comparison — any difference is due to the baseline

def train_reinforce(
    n_episodes: int = 400,
    lr: float = 1e-3,
    gamma: float = 0.99,
    use_baseline: bool = False,
    seed: int = SEED
) -> dict:
    """
    Train REINFORCE on CartPole-v1, optionally with a value baseline.

    Returns
    -------
    dict with keys: 'rewards', 'entropies', 'smoothed_rewards'
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    env = gym.make('CartPole-v1')
    policy = PolicyNetwork(obs_dim, act_dim)
    policy_opt = optim.Adam(policy.parameters(), lr=lr)

    baseline = None
    baseline_opt = None
    if use_baseline:
        baseline = BaselineNetwork(obs_dim)
        baseline_opt = optim.Adam(baseline.parameters(), lr=lr)

    rewards_log, entropy_log = [], []

    for ep in range(n_episodes):
        if use_baseline:
            episode = collect_episode_with_obs(env, policy)
            reinforce_baseline_update(episode, policy, baseline, policy_opt, baseline_opt, gamma)
        else:
            episode = collect_episode(env, policy)
            reinforce_update(episode, policy_opt, gamma)

        rewards_log.append(episode['total_reward'])
        # Mean entropy across the episode
        ep_entropy = torch.stack(episode['entropy']).mean().item()
        entropy_log.append(ep_entropy)

    env.close()

    # Smooth rewards with a window of 20 for cleaner visualization
    window = 20
    smoothed = np.convolve(rewards_log, np.ones(window) / window, mode='valid')

    return {
        'rewards': rewards_log,
        'entropies': entropy_log,
        'smoothed_rewards': smoothed
    }


print("Training REINFORCE (no baseline)...")
results_no_baseline = train_reinforce(n_episodes=400, use_baseline=False)

print("Training REINFORCE (with baseline)...")
results_with_baseline = train_reinforce(n_episodes=400, use_baseline=True)

print(f"No baseline — final 20-ep avg: {np.mean(results_no_baseline['rewards'][-20:]):.1f}")
print(f"With baseline — final 20-ep avg: {np.mean(results_with_baseline['rewards'][-20:]):.1f}")

## 6. Visualizing Results

Three plots tell the full story:
1. **Reward curves** — does the baseline help converge faster?
2. **Policy entropy** — does the policy become less random as training progresses?
3. **Reward variance** — the baseline's main job is variance reduction.

In [ ]:
# Purpose: Visualize training dynamics for both variants
# Key Concept: Smoothed curves reveal learning speed; entropy reveals exploration behavior

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Plot 1: Smoothed reward curves ---
ax = axes[0]
window = 20
x_smooth = np.arange(window - 1, 400)

ax.plot(x_smooth, results_no_baseline['smoothed_rewards'],
        label='No baseline', color='steelblue', linewidth=2)
ax.plot(x_smooth, results_with_baseline['smoothed_rewards'],
        label='With baseline', color='darkorange', linewidth=2)
ax.axhline(y=475, color='green', linestyle='--', alpha=0.7, label='Near-optimal (475)')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Total Reward (20-ep avg)', fontsize=12)
ax.set_title('REINFORCE: Reward Curves', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 2: Policy entropy over training ---
ax = axes[1]
entropy_smooth_nb = np.convolve(results_no_baseline['entropies'],
                                np.ones(window) / window, mode='valid')
entropy_smooth_wb = np.convolve(results_with_baseline['entropies'],
                                np.ones(window) / window, mode='valid')

ax.plot(x_smooth, entropy_smooth_nb, label='No baseline', color='steelblue', linewidth=2)
ax.plot(x_smooth, entropy_smooth_wb, label='With baseline', color='darkorange', linewidth=2)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Policy Entropy (nats)', fontsize=12)
ax.set_title('Policy Entropy Over Training', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 3: Rolling reward standard deviation (variance proxy) ---
ax = axes[2]
window_var = 30

def rolling_std(arr, w):
    return [np.std(arr[max(0, i-w):i+1]) for i in range(len(arr))]

std_nb = rolling_std(results_no_baseline['rewards'], window_var)
std_wb = rolling_std(results_with_baseline['rewards'], window_var)

ax.plot(std_nb, label='No baseline', color='steelblue', linewidth=2, alpha=0.8)
ax.plot(std_wb, label='With baseline', color='darkorange', linewidth=2, alpha=0.8)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Reward Std Dev (30-ep window)', fontsize=12)
ax.set_title('Gradient Variance Proxy', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reinforce_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to reinforce_comparison.png")

## Exercise 6.1: Entropy Bonus

**Task:** Entropy regularization encourages exploration by adding a term to the loss:

$$\mathcal{L}(\theta) = -\sum_t \log \pi_\theta(a_t | s_t) \cdot G_t - \beta \cdot H(\pi)$$

A larger $\beta$ forces the policy to maintain higher entropy (more exploration). Modify `reinforce_update` to include an entropy bonus with coefficient $\beta = 0.01$.

**Expected output:** The modified function returns a loss value and the policy should train normally on CartPole-v1.

<details>
<summary>Hint 1</summary>
Stack the entropy tensors from `episode['entropy']` using `torch.stack()`, then take their mean.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
Subtract `beta * entropy_mean` from your loss (subtract because entropy is positive and we want to maximize it — but we're minimizing the loss).
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Modify reinforce_update to add an entropy bonus term

def reinforce_update_with_entropy(
    episode: dict,
    optimizer: optim.Optimizer,
    gamma: float = 0.99,
    beta: float = 0.01
) -> float:
    """
    REINFORCE update with entropy bonus.

    Parameters
    ----------
    episode : dict
        Episode data from collect_episode().
    optimizer : optim.Optimizer
        Policy network optimizer.
    gamma : float
        Discount factor.
    beta : float
        Entropy bonus coefficient.

    Returns
    -------
    float
        Total loss value.
    """
    your_answer = None  # Replace with your implementation
    return your_answer

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_6_1():
    torch.manual_seed(0)
    env_t = gym.make('CartPole-v1')
    policy_t = PolicyNetwork(obs_dim, act_dim)
    opt_t = optim.Adam(policy_t.parameters(), lr=1e-3)

    ep = collect_episode(env_t, policy_t)
    env_t.close()

    # Test that the function runs and returns a float
    loss = reinforce_update_with_entropy(ep, opt_t, beta=0.01)
    assert loss is not None, "Function must return a loss value, not None"
    assert isinstance(loss, float), f"Expected float, got {type(loss)}"

    # Test that beta=0 gives same result as standard REINFORCE
    torch.manual_seed(1)
    env_t2 = gym.make('CartPole-v1')
    policy_t2 = PolicyNetwork(obs_dim, act_dim)
    opt_t2 = optim.Adam(policy_t2.parameters(), lr=1e-3)
    ep2 = collect_episode(env_t2, policy_t2)
    env_t2.close()
    loss_zero_beta = reinforce_update_with_entropy(ep2, opt_t2, beta=0.0)
    assert loss_zero_beta is not None, "beta=0 case must also return a loss"

    print("Exercise 6.1 passed! Entropy bonus correctly implemented.")

test_exercise_6_1()

## Exercise 6.2: Discount Factor Sensitivity

**Task:** Train REINFORCE (no baseline) with three different discount factors: $\gamma \in \{0.9, 0.95, 0.99\}$. Run each for 300 episodes, then plot all three smoothed reward curves on the same axes.

**Expected output:** A matplotlib figure with 3 labeled curves.

<details>
<summary>Hint 1</summary>
Call `train_reinforce(n_episodes=300, use_baseline=False, gamma=g)` in a loop over gamma values.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
Use different seeds for each run to avoid confounding seed effects: `seed=42`, `seed=43`, `seed=44`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
gammas_to_test = [0.9, 0.95, 0.99]
gamma_results = {}  # Map gamma -> results dict from train_reinforce

# Fill gamma_results with training runs
# Then plot all three smoothed_rewards curves

fig_gamma, ax_gamma = plt.subplots(figsize=(9, 5))
# YOUR PLOTTING CODE HERE
ax_gamma.set_xlabel('Episode')
ax_gamma.set_ylabel('Smoothed Reward')
ax_gamma.set_title('REINFORCE: Effect of Discount Factor')
ax_gamma.legend()
ax_gamma.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_6_2():
    assert len(gamma_results) == 3, \
        f"Expected 3 gamma experiments, got {len(gamma_results)}"
    for g in gammas_to_test:
        assert g in gamma_results, f"Missing results for gamma={g}"
        assert 'rewards' in gamma_results[g], \
            f"Results for gamma={g} missing 'rewards' key"
        assert len(gamma_results[g]['rewards']) >= 100, \
            f"Need at least 100 episodes for gamma={g}, got {len(gamma_results[g]['rewards'])}"
    print("Exercise 6.2 passed! Discount factor comparison complete.")

test_exercise_6_2()

## Key Takeaways

1. **Policy gradient theorem** gives an unbiased gradient estimator by weighting log-probabilities by returns — actions that led to high reward get reinforced.

2. **REINFORCE is high-variance** because it uses a single-sample Monte Carlo estimate of the return. This causes noisy gradients and slow, unstable learning.

3. **Baselines reduce variance** by centering the return signal at each state. The baseline $b(s)$ does not bias the gradient because $\mathbb{E}[\nabla \log \pi \cdot b(s)] = 0$.

4. **Policy entropy** starts high (random policy) and decreases as the policy specializes. An entropy bonus can prevent premature convergence to a suboptimal deterministic policy.

5. **Discount factor $\gamma$** controls how myopic the agent is. Too low and the agent ignores future consequences; too high and early returns dominate gradient estimates.

## What's Next

The baseline network we added is the critic in **Actor-Critic** methods. In `02_actor_critic.ipynb`, we make this coupling tighter: instead of Monte Carlo returns, the critic provides **bootstrapped value estimates** at every step, dramatically reducing variance.

## Additional Resources

- Sutton & Barto, Chapter 13: Policy Gradient Methods
- Williams (1992): "Simple Statistical Gradient-Following Algorithms" — the original REINFORCE paper
- Spinning Up (OpenAI): [Policy Gradient](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html)